In [10]:
import json

import boto3
import pandas as pd
import plotly.express as px

from sqlalchemy import URL, create_engine


AWS_REGION = "eu-west-2"
DB_INSTANCE_ID = "jurassic-sparks-warehouse"
DATABASE_NAME = "jurassic_sparks_warehouse"


# Find the RDS instance and its managed secret.
rds_client = boto3.client(
    "rds",
    region_name=AWS_REGION,
)

db_instance = rds_client.describe_db_instances(
    DBInstanceIdentifier=DB_INSTANCE_ID,
)["DBInstances"][0]

secret_arn = db_instance["MasterUserSecret"]["SecretArn"]


# Retrieve the credentials without displaying them.
secrets_client = boto3.client(
    "secretsmanager",
    region_name=AWS_REGION,
)

response = secrets_client.get_secret_value(
    SecretId=secret_arn,
)

credentials = json.loads(response["SecretString"])


# Build the database connection safely.
database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=credentials["username"],
    password=credentials["password"],
    host=db_instance["Endpoint"]["Address"],
    port=db_instance["Endpoint"]["Port"],
    database=DATABASE_NAME,
)

engine = create_engine(
    database_url,
    connect_args={"sslmode": "require"},
    pool_pre_ping=True,
)

print("Database engine created successfully")

Database engine created successfully


In [11]:
connection_test = pd.read_sql(
    """
    SELECT
        current_database() AS database,
        current_user AS connected_as,
        current_timestamp AS checked_at;
    """,
    engine,
)

connection_test

,database,connected_as,checked_at
0,jurassic_sparks_warehouse,postgres,2026-08-21 08:15:22.379440+00:00


In [12]:
staff_sales = pd.read_sql(
    """
    SELECT
        ds.first_name || ' ' || ds.last_name AS full_name,
        SUM(fs.units_sold) AS total_units_sold
    FROM dim_staff AS ds
    LEFT JOIN fact_sales AS fs
        ON ds.staff_id = fs.sales_staff_id
    GROUP BY
        ds.staff_id,
        ds.first_name,
        ds.last_name
    ORDER BY total_units_sold DESC;
    """,
    engine,
)

staff_sales

,full_name,total_units_sold
0,Imani Walker,286629970.0
1,Magdalena Zieme,279874185.0
2,Stan Lehner,278964040.0
3,Jeremie Franey,277462220.0
4,Rigoberto VonRueden,277306390.0
5,Brody Ratke,277055285.0
6,Deron Beier,276370455.0
7,Tomasa Moore,275571010.0
8,Jeanette Erdman,273804210.0
9,Jett Parisian,273693905.0


In [13]:
chart_data = staff_sales.sort_values(
    "total_units_sold",
    ascending=True,
)

fig = px.bar(
    chart_data,
    x="total_units_sold",
    y="full_name",
    orientation="h",
    title="Total Units Sold by Staff Member",
    labels={
        "full_name": "Staff member",
        "total_units_sold": "Total units sold",
    },
    text_auto=",",
)

fig.update_layout(
    template="plotly_white",
    height=max(500, len(chart_data) * 35),
    xaxis_title="Total units sold",
    yaxis_title="Staff member",
    showlegend=False,
)

fig.update_traces(
    textposition="outside",
)

fig.show()